# 🤖 AI Engineering Fundamentals — Lezione 4
## Notebook Gruppo B

**ITS Novitas 4.0 | Giovedì 28/05/2026**

---

### 📋 Istruzioni
1. **File → Salva una copia in Drive** prima di iniziare
2. Lavorate in gruppo — discutete prima di scrivere
3. Alla fine: **File → Scarica → .ipynb** e caricate su GitHub

### 👥 Membri del gruppo

In [ ]:
GRUPPO = "B"
MEMBRI = ["", "", "", ""]  # ← inserite i vostri nomi
print(f"Gruppo {GRUPPO} — {', '.join(m for m in MEMBRI if m)}")

In [ ]:
# ⚠️ Prima esecuzione: ChromaDB scarica Sentence Transformers (~90MB)
# Le dipendenze (anthropic, chromadb, ...) sono nel requirements.txt
# La API key viene letta dal file .env nella root del progetto (non più dai Secrets di Colab)
import anthropic, os, chromadb
from dotenv import load_dotenv, find_dotenv

load_dotenv(find_dotenv(usecwd=True))   # carica ANTHROPIC_API_KEY dal .env

client = anthropic.Anthropic()          # legge automaticamente ANTHROPIC_API_KEY dall'ambiente
chroma_client = chromadb.Client()

DOCUMENTO_WIDATA = """
WiData Srl — Manuale Prodotti IoT

SENSORE XS200 - MONITORAGGIO AMBIENTALE
Il sensore XS200 è progettato per il monitoraggio ambientale in ambienti industriali e urbani.
Misura temperatura (-20°C a +60°C), umidità relativa (0-100%), pressione atmosferica
e qualità dell'aria (CO2, PM2.5). Classificazione IP67: impermeabile e resistente alla polvere.
Alimentazione: batteria Li-Ion 3.7V, autonomia 2 anni. Connettività: LoRaWAN, NB-IoT, WiFi.
Certificazioni: CE, FCC, RoHS. Garanzia: 3 anni.

GATEWAY GW500 - CONCENTRATORE DATI
Il gateway GW500 raccoglie dati da fino a 1000 sensori simultaneamente tramite LoRaWAN.
Copertura fino a 15km in aree rurali, 3km in aree urbane.
Connessione cloud via Ethernet, WiFi o 4G LTE. Storage locale: 32GB SSD.
Alimentazione: 220V AC o pannello solare. Temperatura operativa: -40°C a +70°C.

PIATTAFORMA XPLORE - ANALYTICS
Xplore è la piattaforma cloud di WiData per visualizzazione e analisi dei dati IoT.
Dashboard personalizzabili con grafici real-time, storico dati fino a 5 anni.
Alerting automatico via email, SMS o webhook.
API REST per integrazione con sistemi terzi (ERP, SCADA, BIM).
Piani: Free (5 sensori), Pro (100 sensori, €49/mese), Enterprise (illimitato).

SUPPORTO E ASSISTENZA
Supporto tecnico disponibile lunedì-venerdì 9:00-18:00.
Email: support@widata.cloud | Telefono: +39 079 123456.
Sede: Via Roma 42, Sassari (SS) 07100, Italia.
"""

print("✅ Setup completato!")

---
## 🎯 Tema del Gruppo B: Chunking & Embedding

Esplorate come la dimensione dei chunk e l'overlap
impattano la qualità del retrieval.
Trovate i parametri ottimali per il documento WiData.

---
### Esercizio 1 — Confronto chunk_size *(guidato)*

Stessa domanda, stesso documento, tre dimensioni di chunk diverse.
I chunk recuperati sono diversi? Quale dimensione trova le informazioni più utili?

In [ ]:
# Esercizio 1 — confronto chunk_size

def chunka_testo(testo, chunk_size=400, overlap=50):
    chunks = []
    start = 0
    while start < len(testo):
        chunk = testo[start:start+chunk_size]
        if chunk.strip():
            chunks.append(chunk)
        start += chunk_size - overlap
    return chunks

def crea_collection(nome, chunks):
    """Crea una collection ChromaDB con i chunk forniti."""
    try:
        chroma_client.delete_collection(nome)
    except:
        pass
    coll = chroma_client.create_collection(nome)
    coll.add(documents=chunks, ids=[str(i) for i in range(len(chunks))])
    return coll

def cerca_in_collection(domanda, collection, n=2):
    risultati = collection.query(query_texts=[domanda], n_results=n)
    return risultati["documents"][0]

# Crea tre collection con dimensioni diverse
dimensioni = [100, 400, 800]
collections = {}

for dim in dimensioni:
    chunks = chunka_testo(DOCUMENTO_WIDATA, chunk_size=dim, overlap=20)
    collections[dim] = crea_collection(f"widata_b_{dim}", chunks)
    print(f"chunk_size={dim}: {len(chunks)} chunk generati")

print()

# Stessa domanda sulle tre collection
domanda = "Qual è l'autonomia della batteria del sensore XS200?"
print(f"❓ {domanda}\n")

for dim in dimensioni:
    print(f"{'='*50}")
    print(f"chunk_size = {dim}")
    # Cerchiamo nella collection corrispondente a questa dimensione di chunk
    chunks_trovati = cerca_in_collection(domanda, collections[dim])
    for i, chunk in enumerate(chunks_trovati):
        print(f"Chunk {i+1}: {chunk[:150]}...")
    print()

# Osservazione: chunk troppo piccoli (100) spezzano la frase e a volte perdono il
# contesto ("autonomia 2 anni" può finire diviso); chunk troppo grandi (800) includono
# molto testo non pertinente. Una dimensione intermedia (~400) di solito recupera il
# blocco giusto del sensore XS200 mantenendo l'informazione completa e poco rumore.

---
### Esercizio 2 — L'effetto dell'overlap *(guidato)*

Spezzate lo stesso testo con overlap=0 e overlap=100.
Trovate un caso in cui la frase chiave è a cavallo tra due chunk
e verificate che l'overlap la recuperi correttamente.

In [ ]:
# Esercizio 2 — effetto dell'overlap

# Testo di test dove l'informazione chiave è a cavallo tra due chunk
testo_test = """Il sensore XS200 è resistente alle intemperie grazie alla classificazione
IP67 che garantisce impermeabilità completa e protezione dalla polvere.
La connettività LoRaWAN permette trasmissioni fino a 15km in campo aperto."""

# Chunk size piccolo per forzare lo spezzamento nel punto critico
CHUNK_SIZE = 80

# SENZA overlap
chunks_no_overlap = chunka_testo(testo_test, chunk_size=CHUNK_SIZE, overlap=0)
print("SENZA overlap:")
for i, c in enumerate(chunks_no_overlap):
    print(f"  Chunk {i+1}: '{c}'")

print()

# CON overlap
chunks_overlap = chunka_testo(testo_test, chunk_size=CHUNK_SIZE, overlap=30)
print("CON overlap=30:")
for i, c in enumerate(chunks_overlap):
    print(f"  Chunk {i+1}: '{c}'")

print()

# Creiamo due collection e confrontiamo il chunk recuperato
coll_no = crea_collection("test_no_overlap", chunks_no_overlap)
coll_si = crea_collection("test_overlap", chunks_overlap)

domanda_test = "Il sensore è impermeabile e come trasmette i dati?"

print(f"❓ {domanda_test}")
print("\nSENZA overlap — chunk recuperato:")
print(cerca_in_collection(domanda_test, coll_no, n=1)[0])
print("\nCON overlap — chunk recuperato:")
print(cerca_in_collection(domanda_test, coll_si, n=1)[0])

# Osservazione: senza overlap il punto in cui si parla di "impermeabilità" e quello
# della "connettività LoRaWAN" possono finire in chunk diversi, così il singolo chunk
# recuperato copre solo metà della risposta. Con overlap i chunk si sovrappongono e
# hanno più probabilità di contenere insieme i concetti vicini. L'overlap è più
# importante quando informazioni correlate stanno a cavallo del confine tra due chunk.

---
### Esercizio 3 — Trovare i parametri ottimali *(libero)*

Create un mini-benchmark: 5 domande sul documento WiData
con risposte attese note. Testate almeno 4 combinazioni
di chunk_size e overlap. Quale combinazione risponde
correttamente al maggior numero di domande?

In [ ]:
# Esercizio 3 — benchmark parametri chunking

# Dataset di test: domanda + risposta attesa (parola chiave)
dataset = [
    {"domanda": "Qual è l'autonomia del sensore XS200?", "atteso": "2 anni"},
    {"domanda": "Quanti sensori gestisce il gateway GW500?", "atteso": "1000"},
    {"domanda": "Quanto costa il piano Pro di Xplore?", "atteso": "49"},
    {"domanda": "Qual è il numero di telefono del supporto?", "atteso": "079"},
    {"domanda": "Qual è la classificazione IP del sensore?", "atteso": "IP67"},
]

# Combinazioni da testare
configurazioni = [
    {"chunk_size": 100, "overlap": 10},
    {"chunk_size": 200, "overlap": 30},
    {"chunk_size": 400, "overlap": 50},
    {"chunk_size": 800, "overlap": 100},
]

print(f"{'Config':<25} {'Corrette/5':<15} {'Score'}")
print("-" * 50)

risultati_bench = {}
for conf in configurazioni:
    cs = conf["chunk_size"]
    ov = conf["overlap"]

    # Indicizziamo il documento con questa configurazione
    chunks = chunka_testo(DOCUMENTO_WIDATA, chunk_size=cs, overlap=ov)
    coll = crea_collection(f"bench_{cs}_{ov}", chunks)

    # Per ogni domanda verifichiamo se la parola attesa è nei chunk recuperati
    corrette = 0
    for item in dataset:
        trovati = cerca_in_collection(item["domanda"], coll, n=2)
        contesto = " ".join(trovati)
        if item["atteso"].lower() in contesto.lower():
            corrette += 1

    risultati_bench[(cs, ov)] = corrette
    print(f"size={cs}, overlap={ov:<10} {corrette}/5")

# Migliore configurazione (a parità di score, la prima trovata)
migliore = max(risultati_bench, key=risultati_bench.get)
print()
print(f"Configurazione ottimale per WiData: chunk_size={migliore[0]}, overlap={migliore[1]}")
print("Motivazione: massimizza il numero di risposte trovate nel retrieval; chunk di "
      "dimensione media con overlap sufficiente catturano l'informazione completa senza "
      "introdurre troppo rumore (e senza spezzare le frasi chiave).")

---
### Esercizio 4 — Chunking semantico *(libero)*

Il chunking a lunghezza fissa è semplice ma spezza le frasi.
Implementate una versione che spezza sui paragrafi naturali
del documento. Confrontate con il chunking a lunghezza fissa.

In [ ]:
# Esercizio 4 — chunking semantico sui paragrafi

def chunka_per_paragrafi(testo, max_chunk_size=600):
    """
    Spezza il testo sui paragrafi (doppio newline).
    Se un paragrafo è troppo lungo, lo spezza ulteriormente.
    """
    chunks = []
    for paragrafo in testo.split("\n\n"):
        p = paragrafo.strip()
        if not p:
            continue  # salta i chunk vuoti
        if len(p) <= max_chunk_size:
            chunks.append(p)
        else:
            # paragrafo troppo lungo: lo spezziamo in pezzi da max_chunk_size
            for start in range(0, len(p), max_chunk_size):
                pezzo = p[start:start + max_chunk_size].strip()
                if pezzo:
                    chunks.append(pezzo)
    return chunks

chunks_semantici = chunka_per_paragrafi(DOCUMENTO_WIDATA)
chunks_fissi = chunka_testo(DOCUMENTO_WIDATA, chunk_size=400, overlap=50)

print(f"Chunking fisso:    {len(chunks_fissi)} chunk")
print(f"Chunking semantico: {len(chunks_semantici)} chunk")
print()

# Mostrate i chunk semantici
print("Chunk semantici:")
for i, c in enumerate(chunks_semantici):
    print(f"  Chunk {i+1} ({len(c)} char): {c[:80]}...")

print()

# Confronto retrieval sui due tipi di chunking con le 5 domande del benchmark
coll_fissi = crea_collection("cmp_fissi", chunks_fissi)
coll_sem = crea_collection("cmp_semantici", chunks_semantici)

print(f"{'Domanda':<45} {'Fisso':<8} {'Semantico'}")
print("-" * 65)
for item in dataset:
    ctx_f = " ".join(cerca_in_collection(item["domanda"], coll_fissi, n=2))
    ctx_s = " ".join(cerca_in_collection(item["domanda"], coll_sem, n=2))
    ok_f = "✅" if item["atteso"].lower() in ctx_f.lower() else "❌"
    ok_s = "✅" if item["atteso"].lower() in ctx_s.lower() else "❌"
    print(f"{item['domanda'][:43]:<45} {ok_f:<8} {ok_s}")

# Osservazione: il chunking semantico segue i paragrafi naturali (un blocco per prodotto),
# quindi ogni chunk contiene un'informazione coerente e completa: spesso migliora il
# retrieval su questo documento ben strutturato. Il chunking fisso è più semplice e
# generico ma può spezzare le sezioni a metà. Il semantico è preferibile quando il
# documento ha una struttura chiara (titoli/paragrafi); il fisso quando il testo è
# omogeneo e senza una struttura evidente.

---
## 📊 Preparate la presentazione (5 slide)

1. **Chunk troppo piccoli vs ottimali vs grandi** — con i vostri esempi concreti
2. **L'effetto dell'overlap** — mostrate il caso della frase spezzata
3. **Il vostro benchmark** — tabella con i risultati delle 4 configurazioni
4. **Chunking semantico vs fisso** — differenze e quando usare quale
5. **La vostra raccomandazione** — parametri ottimali per WiData con motivazione

---
*ITS Novitas 4.0 — AI Engineering Fundamentals | Marco Uras*